# ReasonLite — Notebook 2: controlled GRPO pilot
**E2 correctness-only vs E3 correctness + efficiency + format shaping.** Both start independently from the exact E1 LoRA checkpoint. 64 frozen RL prompts per arm, 4 completions per prompt, single GPU, one epoch. This is a small single-seed pilot, not a definitive efficacy study. No official test evaluation. Training cap 256, evaluation cap stays 384. TRL 0.17 uses beta=0 here (no KL reference) to isolate the reward and avoid an adapter-disabled base reference. Output length in the reward is approximate re-tokenized decoded text; evaluation uses real generated IDs. Keep the archive and notebook private.

If your Notebook 1 Kaggle session is still open, append these code cells there; otherwise attach the Notebook 1 ZIP as a private Kaggle dataset via Add Input, enable GPU and Internet, then run in order. Do not rerun baseline inference.

In [ ]:
import os, sys, subprocess, json, zipfile, hashlib, shutil
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
WORK = Path('/kaggle/working')
ROOT = WORK/'ReasonLite'
assert WORK.is_dir(), 'Kaggle environment required'
os.environ.update(CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false', WANDB_DISABLED='true', PYTHONUNBUFFERED='1', HF_HUB_DISABLE_TELEMETRY='1')
# Reuse the live session or restore the exact Notebook 1 archive.
if not (ROOT/'runs/e1_sft_v2/adapter/adapter_model.safetensors').is_file():
    candidates = list(Path('/kaggle/input').rglob('ReasonLite_Notebook1_Complete.zip')) + list(WORK.glob('ReasonLite_Notebook1_Complete.zip'))
    assert candidates, 'Attach ReasonLite_Notebook1_Complete.zip as private Kaggle input'
    archive=candidates[0]
    ROOT.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            if item.is_dir(): continue
            dest=ROOT/item.filename
            assert dest.resolve().is_relative_to(ROOT.resolve())
            payload=z.read(item)
            if dest.exists(): assert dest.read_bytes()==payload, f'Existing file differs: {dest}'
            else: dest.parent.mkdir(parents=True,exist_ok=True); dest.write_bytes(payload)
os.chdir(ROOT)
def run(*args):
    subprocess.run([sys.executable,'-u',*map(str,args)],cwd=ROOT,check=True)
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(ROOT/'data/prepared/manifest.json')=='3fcde5df1054b269cc4aa4b755e499848564f62d4532e6acde9379e653aa533c'
assert sha(ROOT/'runs/e1_sft_v2/adapter/adapter_model.safetensors')=='5d885ada194f925c2f0ab34a25c503679e47c450b8d32009e775d34c2528360b'
run('-c', "import torch; assert torch.cuda.is_available() and torch.cuda.device_count()==1; print(torch.__version__, torch.cuda.get_device_name(0))")
print('Verified E1 checkpoint and manifest')

In [ ]:
requirements=['transformers==4.51.3','datasets==3.5.0','huggingface-hub==0.30.2','accelerate==1.6.0','trl==0.17.0','peft==0.15.2']
missing=[]
for req in requirements:
    name, want=req.split('==')
    try: actual=version(name)
    except PackageNotFoundError: actual=None
    if actual!=want: missing.append(req)
if missing: run('-m','pip','install','--upgrade-strategy','only-if-needed',*missing)
else: print('Pinned dependencies already installed')
run('-c','from trl import GRPOConfig, GRPOTrainer; import peft, torch; print("GRPO imports successful")')
run('-m','unittest','discover','-s','tests','-v')

In [ ]:
source = '"""Low-compute TRL 0.17 GRPO pilot: independent arms from a frozen SFT adapter."""\nimport argparse, json, math, time, platform\nfrom fractions import Fraction\nfrom pathlib import Path\nfrom .data import load_rows, digest, dump_json, messages\nfrom .scoring import grade, SCORING_VERSION\n\nEXPECTED_PARENT = \'5d885ada194f925c2f0ab34a25c503679e47c450b8d32009e775d34c2528360b\'\nEXPECTED_MANIFEST = \'3fcde5df1054b269cc4aa4b755e499848564f62d4532e6acde9379e653aa533c\'\n\ndef shape(text, gold, tokens, mode, cap=256, budget=128):\n    if mode not in (\'correctness\',\'efficiency\') or tokens < 0 or budget <= 0:\n        raise ValueError(\'Invalid reward configuration\')\n    # TRL 0.17 callback supplies decoded text, not generation token IDs.\n    # Near-cap re-tokenized outputs are treated conservatively by the fallback scorer.\n    g = grade(text, Fraction(gold), truncated=tokens >= cap-1)\n    value = 0.0\n    if g[\'correct\']:\n        value = 1.0\n        if mode == \'efficiency\':\n            value -= .15 * min(1., max(0,tokens-budget)/budget)\n            value += .05 * g[\'format_ok\']\n    return float(value), g\n\ndef main():\n    p=argparse.ArgumentParser(); p.add_argument(\'--data\',default=\'data/prepared\'); p.add_argument(\'--parent\',default=\'runs/e1_sft_v2/adapter\'); p.add_argument(\'--out\',required=True); p.add_argument(\'--mode\',choices=[\'correctness\',\'efficiency\'],required=True); p.add_argument(\'--prompts\',type=int,default=64); p.add_argument(\'--smoke\',action=\'store_true\'); a=p.parse_args()\n    import torch, transformers, trl, peft\n    from datasets import Dataset\n    from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed\n    from trl import GRPOConfig, GRPOTrainer\n    from peft import PeftModel\n    from safetensors.torch import load_file\n    assert torch.cuda.is_available() and torch.cuda.device_count()==1, \'Enable one visible CUDA GPU\'\n    assert trl.__version__==\'0.17.0\' and transformers.__version__==\'4.51.3\' and peft.__version__==\'0.15.2\', \'Use the pinned environment\'\n    data=Path(a.data); parent=Path(a.parent); manifest,rl=load_rows(data,\'rl\')\n    assert SCORING_VERSION==\'final-answer-v2\'\n    assert digest(data/\'manifest.json\')==EXPECTED_MANIFEST\n    assert digest(parent/\'adapter_model.safetensors\')==EXPECTED_PARENT\n    for split in (\'sft\',\'validation\'):\n        _, other=load_rows(data,split); assert not ({r[\'id\'] for r in rl}&{r[\'id\'] for r in other})\n    assert 1<=a.prompts<=len(rl)\n    rows=rl[:a.prompts]\n    parent_meta=json.loads((parent/\'reasonlite_training.json\').read_text())\n    assert parent_meta[\'model_revision\']==manifest[\'model_revision\'] and parent_meta[\'data_manifest_sha256\']==EXPECTED_MANIFEST\n    set_seed(manifest[\'seed\']); out=Path(a.out); out.mkdir(parents=True,exist_ok=False)\n    tok=AutoTokenizer.from_pretrained(manifest[\'model\'],revision=manifest[\'model_revision\'],trust_remote_code=False)\n    tok.padding_side=\'left\'\n    if tok.pad_token_id is None: tok.pad_token=tok.eos_token\n    for row in rows:\n        ids=tok.apply_chat_template(messages(row[\'question\']),tokenize=True,add_generation_prompt=True)\n        assert len(ids)<=512, \'Prompt would be truncated: \'+row[\'id\']\n    model=AutoModelForCausalLM.from_pretrained(manifest[\'model\'],revision=manifest[\'model_revision\'],torch_dtype=torch.float32,trust_remote_code=False)\n    model=PeftModel.from_pretrained(model,str(parent),is_trainable=True)\n    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)\n    total=sum(p.numel() for p in model.parameters())\n    assert trainable==parent_meta[\'trainable_parameters\'] and 0<trainable<total\n    assert all(p.dtype==torch.float32 for p in model.parameters() if p.requires_grad)\n    dataset=Dataset.from_list([{\'prompt\':messages(r[\'question\']),\'gold\':r[\'gold\'],\'prompt_id\':r[\'id\']} for r in rows])\n    count={\'rollouts\':0,\'correct\':0,\'valid\':0,\'format_ok\':0,\'tokens\':0,\'reward_sum\':0.0,\'groups\':0,\'nonconstant_reward_groups\':0}\n    cap=256\n    def reward_fn(prompts, completions, gold, prompt_id, **kwargs):\n        values=[]; records=[]\n        assert len(completions)==len(gold)==len(prompt_id)==len(prompts)\n        for completion, target, pid in zip(completions,gold,prompt_id):\n            text=completion if isinstance(completion,str) else completion[0][\'content\']\n            n=len(tok(text,add_special_tokens=False)[\'input_ids\'])\n            value,g=shape(text,target,n,a.mode,cap)\n            values.append(value); records.append({\'id\':pid,\'gold\':target,\'output\':text,\'retokenized_tokens\':n,\'reward\':value,**g})\n            count[\'rollouts\']+=1; count[\'correct\']+=int(g[\'correct\']); count[\'valid\']+=int(g[\'valid\']); count[\'format_ok\']+=int(g[\'format_ok\']); count[\'tokens\']+=n; count[\'reward_sum\']+=value\n        # Single process, batch size == generations; verify grouping explicitly.\n        assert len(records)==4 and len({r[\'id\'] for r in records})==1, \'Unexpected rollout grouping\'\n        count[\'groups\']+=1; count[\'nonconstant_reward_groups\']+=int(max(values)-min(values)>1e-8)\n        with (out/\'rollouts.jsonl\').open(\'a\',encoding=\'utf-8\') as f:\n            for r in records: f.write(json.dumps(r,ensure_ascii=False,allow_nan=False)+\'\\n\')\n        return values\n    config=dict(per_device_train_batch_size=4,gradient_accumulation_steps=1,num_generations=4,max_prompt_length=512,max_completion_length=cap,\n        learning_rate=2e-5,num_train_epochs=1,max_steps=1 if a.smoke else -1,temperature=0.8,top_p=1.0,top_k=None,\n        beta=0.0,num_iterations=1,epsilon=0.2,loss_type=\'bnpo\',scale_rewards=True,mask_truncated_completions=False,\n        gradient_checkpointing=True,gradient_checkpointing_kwargs={\'use_reentrant\':False},disable_dropout=True,fp16=True,bf16=False,\n        max_grad_norm=1.0,lr_scheduler_type=\'constant\',logging_steps=1,save_strategy=\'steps\',save_steps=8,save_total_limit=1,\n        report_to=\'none\',seed=manifest[\'seed\'],data_seed=manifest[\'seed\'],shuffle_dataset=True,use_vllm=False,remove_unused_columns=False)\n    args=GRPOConfig(output_dir=str(out/\'checkpoints\'),**config)\n    trainer=GRPOTrainer(model=model,reward_funcs=reward_fn,args=args,train_dataset=dataset,processing_class=tok)\n    torch.cuda.reset_peak_memory_stats(); start=time.perf_counter(); result=trainer.train(); torch.cuda.synchronize(); elapsed=time.perf_counter()-start\n    assert math.isfinite(result.metrics[\'train_loss\']), \'Non-finite training loss\'\n    assert trainer.state.global_step==(1 if a.smoke else a.prompts), (trainer.state.global_step,a.prompts)\n    adapter=out/\'adapter\'; trainer.save_model(str(adapter)); tok.save_pretrained(str(adapter))\n    weights=load_file(str(adapter/\'adapter_model.safetensors\')); initial=load_file(str(parent/\'adapter_model.safetensors\'))\n    assert weights.keys()==initial.keys()\n    assert all(torch.isfinite(w).all().item() for w in weights.values())\n    changed=sum(int((weights[k].float()!=initial[k].float()).sum().item()) for k in weights)\n    meta={\'model_revision\':manifest[\'model_revision\'],\'data_manifest_sha256\':EXPECTED_MANIFEST,\'parent_adapter_sha256\':EXPECTED_PARENT,\n        \'adapter_sha256\':digest(adapter/\'adapter_model.safetensors\'),\'rl_split_sha256\':manifest[\'files\'][\'rl\'][\'sha256\'],\'prompt_ids\':[r[\'id\'] for r in rows],\n        \'mode\':a.mode,\'smoke\':a.smoke,\'scoring_version\':SCORING_VERSION,\'configuration\':config,\'seed\':manifest[\'seed\'],\'optimizer_steps\':trainer.state.global_step,\n        \'trainable_parameters\':trainable,\'total_parameters\':total,\'changed_adapter_values\':changed,\'train_seconds\':elapsed,\n        \'peak_allocated_gpu_mib\':torch.cuda.max_memory_allocated()/2**20,\'gpu\':torch.cuda.get_device_name(0),\'torch\':torch.__version__,\'transformers\':transformers.__version__,\'trl\':trl.__version__,\'peft\':peft.__version__,\'python\':platform.python_version(),\n        \'trainer_metrics\':result.metrics,\'rollout_counts\':count,\'reward_length_measure\':\'decoded-text retokenization (not exact generated token count)\',\'reference_kl\':\'disabled (beta=0)\'}\n    dump_json(adapter/\'reasonlite_training.json\',meta); dump_json(out/\'training_log.json\',trainer.state.log_history); dump_json(out/\'train_metrics.json\',meta)\n    print(json.dumps(meta,indent=2))\n    if not a.smoke and changed==0: print(\'WARNING: no adapter values changed; investigate zero reward variance or skipped updates before interpreting the run.\')\n\nif __name__==\'__main__\': main()\n'
p=ROOT/'reasonlite/train_grpo.py'
if p.exists(): assert p.read_text(encoding='utf-8')==source, 'Existing GRPO source differs'
else: p.write_text(source,encoding='utf-8')
run('-m','py_compile',str(p))
run('-c', "from reasonlite.train_grpo import shape; assert shape('#### 18','18',30,'correctness')[0]==1; assert shape('#### 19','18',1,'efficiency')[0]==0; assert shape('#### 18','18',30,'efficiency')[0]>shape('#### 18','18',250,'efficiency')[0]>0; print('Reward checks passed')")

## Run settings and smoke test
Run the smoke test first. The pilot uses the first 64 of the already seeded 256 RL rows. For an even smaller pilot use 32 **before** running either full arm. Both must use the same number. A partial output directory is never silently reused; rename it before retrying. No automatic checkpoint resume is implemented.

In [ ]:
N_PROMPTS=64  # Set 32 before BOTH arms if compute is limited.
assert N_PROMPTS in (32,64,128,256)
PARENT=ROOT/'runs/e1_sft_v2/adapter'
def train_arm(name,mode,smoke=False):
    dest=ROOT/'runs'/name
    if (dest/'train_metrics.json').exists():
        m=json.loads((dest/'train_metrics.json').read_text())
        assert m['mode']==mode and m['smoke']==smoke and len(m['prompt_ids'])==N_PROMPTS
        assert m['parent_adapter_sha256']==sha(PARENT/'adapter_model.safetensors')
        assert m['adapter_sha256']==sha(dest/'adapter/adapter_model.safetensors')
        print('Reusing verified completed run',dest)
    else:
        assert not dest.exists(), f'Partial run at {dest}; rename it before retrying'
        args=['-m','reasonlite.train_grpo','--parent',str(PARENT),'--out',str(dest),'--mode',mode,'--prompts',str(N_PROMPTS)]
        if smoke: args+=['--smoke']
        run(*args)
        m=json.loads((dest/'train_metrics.json').read_text())
    print({k:m[k] for k in ('optimizer_steps','changed_adapter_values','train_seconds','rollout_counts')})
    return dest
smoke=train_arm('grpo_smoke_v1','correctness',True)
print('GRPO SMOKE PASSED; zero change in a single group is possible')

## E2 — correctness-only GRPO
Correct answer = 1, wrong/invalid = 0. The E1 checkpoint is loaded from disk; smoke training does not initialize this run.

In [ ]:
e2=train_arm('e2_grpo_correctness_v1','correctness')

## E3 — efficiency-aware GRPO
Same parent, prompt IDs, seed, optimizer and generation settings. Wrong/invalid answers = 0. Correct answers get 1 minus a bounded 0–0.15 length penalty above 128 re-tokenized tokens, plus 0.05 for requested answer format. This tests the combined shaping package, not an isolated length-only effect.

In [ ]:
e3=train_arm('e3_grpo_efficiency_v1','efficiency')
m2=json.loads((e2/'train_metrics.json').read_text()); m3=json.loads((e3/'train_metrics.json').read_text())
for key in ('parent_adapter_sha256','prompt_ids','configuration','seed','optimizer_steps','rl_split_sha256','scoring_version'):
    assert m2[key]==m3[key],key
for m in (m2,m3):
    if not m['changed_adapter_values']: print('WARNING: no weight change in',m['mode'])
print('Matched training protocols verified')

## Validation — evaluate E2 and E3 with the original protocol
Same 128 examples, greedy decoding, 384 generated token cap, single example at a time, frozen v2 scoring. No held-out test. If the fresh Kaggle environment has a different torch version/GPU than E0/E1, accuracy comparisons can be descriptive but the strict comparison script will refuse the unmatched protocol. Do not edit metadata to bypass this.

In [ ]:
def evaluate_arm(train_dir,name):
    dest=ROOT/'runs'/name
    if not (dest/'metrics.json').exists():
        assert not dest.exists(), f'Partial evaluation at {dest}; rename it before retrying'
        run('-m','reasonlite.evaluate','--split','validation','--limit','128','--max-new-tokens','384','--device','cuda','--adapter',str(train_dir/'adapter'),'--out',str(dest))
    m=json.loads((dest/'metrics.json').read_text())
    assert m['scoring_version']=='final-answer-v2' and m['n']==128
    assert m['adapter_sha256']==sha(train_dir/'adapter/adapter_model.safetensors')
    return dest
v2=evaluate_arm(e2,'e2_validation_v1')
v3=evaluate_arm(e3,'e3_validation_v1')

In [ ]:
from IPython.display import display, FileLink
paths=[ROOT/'runs/e0_validation_v2',ROOT/'runs/e1_validation_v2',v2,v3]
metrics=[json.loads((p/'metrics.json').read_text()) for p in paths]
keys=['accuracy','strict_accuracy','format_rate','invalid_rate','mean_tokens','correct_per_1000_generated_tokens','mean_latency_s','p95_latency_s','token_limit_rate','peak_allocated_gpu_mib']
print(f'{"Metric":36} {"E0":>12} {"E1":>12} {"E2":>12} {"E3":>12}')
for k in keys: print(f'{k:36} '+ ' '.join(f'{m[k]:12.4f}' for m in metrics))
comparisons=[]
for i,j in [(0,1),(1,2),(1,3),(2,3),(0,2),(0,3)]:
    dest=ROOT/'runs'/f'compare_e{i}_e{j}_nb2.json'
    try:
        run('-m','reasonlite.compare',str(paths[i]),str(paths[j]),'--out',str(dest)); comparisons.append(dest)
    except subprocess.CalledProcessError:
        print(f'Comparison E{i}/E{j} blocked by protocol guard; retain evidence and report mismatch.')
# Retain source, frozen train/validation evidence, all adapters, and logs; no optimizer checkpoints.
export=WORK/'ReasonLite_Notebook2_Complete.zip'
folders=[ROOT/'reasonlite',ROOT/'tests',ROOT/'configs',ROOT/'docs',ROOT/'data/prepared',ROOT/'runs/e0_validation',*paths,ROOT/'runs/e1_sft_v2',smoke,e2,e3]
seen=set()
with zipfile.ZipFile(export,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        for f in folder.rglob('*'):
            if not f.is_file(): continue
            rel=f.relative_to(ROOT)
            if 'checkpoints' in rel.parts or '__pycache__' in rel.parts or rel in seen: continue
            z.write(f,rel); seen.add(rel)
    for f in comparisons+[ROOT/'README.md',ROOT/'requirements.txt']:
        if f.is_file(): z.write(f,f.relative_to(ROOT))
print('Notebook 2 evidence:',export,'MiB:',round(export.stat().st_size/2**20,2))
display(FileLink(str(export)))

### Interpretation rules
A useful outcome is measured accuracy recovery with retained efficiency, not merely increased training reward. Compare E2 vs E3 and both vs E1/E0; inspect paired transitions and reward variance. No claim of statistical superiority from a single small validation run. Preserve all runs including negative outcomes. The official GSM8K test remains reserved for a later frozen protocol.